# NB4 — Hierarchical Cascade Classifier

Trains and evaluates a two-stage cascade classifier for ICS anomaly detection:

- **Stage 1** — Random Forest: Normal vs Anomaly (attack + fault combined)
- **Stage 2** — Random Forest: Attack vs Fault (trained on anomaly rows only)

Outputs: per-class recall curves, per-fault-type recall breakdown, confusion matrix, FPR budget sensitivity table, and saved model artifacts.

**Prerequisites:** Run NB1 → NB2 → NB3 first.

In [1]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

pd.set_option("display.max_columns", 40)

DATA_DIR    = Path("data")
MODELS_DIR  = DATA_DIR / "models"
RESULTS_DIR = DATA_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
for d in [MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RF_N_ESTIMATORS = 300
RF_RANDOM_STATE = 42
FPR_BUDGETS     = [0.005, 0.010, 0.020]  # 0.5%, 1.0%, 2.0%

df = pd.read_parquet(DATA_DIR / "features.parquet")
feat_ref     = json.loads((DATA_DIR / "feature_cols.json").read_text())
FEATURE_COLS = feat_ref["feature_cols"]

print(f"Input:   features.parquet")
print(f"Models  → {MODELS_DIR}/")
print(f"Results → {RESULTS_DIR}/")
print(f"Loaded: {df.shape}")
print(f"Feature columns: {len(FEATURE_COLS)}")
print(f"\nLabel counts:")
for lv, ln in [(0,"normal"),(1,"attack"),(2,"fault")]:
    print(f"  {ln} ({lv}): {(df['label']==lv).sum():>9,}")


Input:   features.parquet
Models  → data/models/
Results → data/results/
Loaded: (954397, 1517)
Feature columns: 1513

Label counts:
  normal (0):   757,798
  attack (1):     9,977
  fault (2):   186,622


## 1. Train/Test Split

Partitions `features.parquet` by the `split` column written by NB2:

- **Stage 1 labels** — binary: 0 = normal, 1 = anomaly (attack or fault combined)
- **Stage 2 labels** — binary: 0 = fault, 1 = attack (trained on anomaly rows only)
- **Original labels** — 3-class ground truth retained as `y_test_orig` for cascade evaluation

In [2]:
train_df = df[df["split"] == "train"].copy()
test_df  = df[df["split"] == "test"].copy()

X_train = train_df[FEATURE_COLS].values
X_test  = test_df[FEATURE_COLS].values

y_train_s1 = (train_df["label"] > 0).astype(int).values
y_test_s1  = (test_df["label"] > 0).astype(int).values

y_test_orig = test_df["label"].values
fault_type_test = test_df["fault_type"].values if "fault_type" in test_df.columns else None

print(f"Train rows: {len(train_df):,}  (anomaly={y_train_s1.sum():,}  normal={(~y_train_s1.astype(bool)).sum():,})")
print(f"Test rows:  {len(test_df):,}  (normal={(y_test_orig==0).sum():,}  attack={(y_test_orig==1).sum():,}  fault={(y_test_orig==2).sum():,})")

Train rows: 772,812  (anomaly=159,132  normal=613,680)
Test rows:  181,585  (normal=144,118  attack=1,914  fault=35,553)


## 2. Stage 1 — Normal vs Anomaly RF

Trains a 300-tree Random Forest on binary (normal / anomaly) labels. Full bootstrap on ~623k rows takes ~60–80 minutes.

- **Class imbalance** — anomaly rows are ~2.6% of train; the RF handles this without reweighting (full bootstrap with majority-class trees)
- **OOB score** — quick sanity check; the threshold for deployment is determined in Section 3 via FPR budget sweep

In [3]:
print("Training Stage 1 RF (normal vs anomaly)...")
rf_s1 = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    random_state=RF_RANDOM_STATE,
    oob_score=True,
    n_jobs=-1,
)
rf_s1.fit(X_train, y_train_s1)
print(f"Stage 1 OOB score: {rf_s1.oob_score_:.4f}")

p_s1 = rf_s1.predict_proba(X_test)[:, 1]  # P(anomaly)


Training Stage 1 RF (normal vs anomaly)...
Stage 1 OOB score: 0.9995


## 3. Stage 1 Threshold Sweep — Operating Points

Sweeps 500 thresholds over `P(anomaly)` to find the highest fault recall achievable within each FPR budget (0.5%, 1.0%, 2.0%). **Stage 1 fault recall is the cascade bottleneck** — faults not passed to Stage 2 are irrecoverable regardless of Stage 2 performance.

In [4]:
n_normal_test  = (y_test_orig == 0).sum()
n_attack_test  = (y_test_orig == 1).sum()
n_fault_test   = (y_test_orig == 2).sum()

sweep_thresholds = np.linspace(0.01, 0.99, 500)
results_s1 = []

for t in sweep_thresholds:
    pred_anomaly = p_s1 >= t
    fp     = ((y_test_orig == 0) & pred_anomaly).sum()
    fpr    = fp / n_normal_test if n_normal_test > 0 else 0
    atk_r  = ((y_test_orig == 1) & pred_anomaly).sum() / n_attack_test if n_attack_test > 0 else 0
    flt_r  = ((y_test_orig == 2) & pred_anomaly).sum() / n_fault_test  if n_fault_test  > 0 else 0
    results_s1.append({"threshold": t, "fpr": fpr, "attack_recall": atk_r, "fault_recall": flt_r})

s1_df = pd.DataFrame(results_s1)

print("Stage 1 Operating Points:")
print(f"{'FPR Budget':>12}  {'Threshold':>10}  {'Fault Recall':>13}  {'Attack Recall':>14}  {'Normal FPR':>11}")
operating_points = {}
for budget in FPR_BUDGETS:
    feasible = s1_df[s1_df["fpr"] <= budget]
    if feasible.empty:
        print(f"  {budget:.1%}  — no feasible threshold")
        continue
    row = feasible.loc[feasible["fault_recall"].idxmax()]
    operating_points[budget] = row
    print(f"  {budget:.1%}  t={row['threshold']:.3f}  fault_recall={row['fault_recall']:.1%}  "
          f"attack_recall={row['attack_recall']:.1%}  fpr={row['fpr']:.2%}")


Stage 1 Operating Points:
  FPR Budget   Threshold   Fault Recall   Attack Recall   Normal FPR
  0.5%  t=0.631  fault_recall=48.1%  attack_recall=75.0%  fpr=0.48%
  1.0%  t=0.530  fault_recall=70.2%  attack_recall=88.5%  fpr=0.99%
  2.0%  t=0.418  fault_recall=81.7%  attack_recall=95.4%  fpr=1.96%


## 4. Stage 2 — Attack vs Fault RF

Trains a second 300-tree Random Forest on **anomaly rows only**, distinguishing attack from fault. Training on a pure anomaly subset eliminates class imbalance against normal rows, giving the classifier a clean signal for the harder attack/fault boundary.

In [5]:
print("Training Stage 2 RF (attack vs fault, anomaly rows only)...")
anomaly_mask_train = train_df["label"].isin([1, 2])
train_s2_df = train_df[anomaly_mask_train].copy()
X_train_s2  = train_s2_df[FEATURE_COLS].values
y_train_s2  = (train_s2_df["label"] == 1).astype(int).values  # 1=attack, 0=fault

rf_s2 = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    random_state=RF_RANDOM_STATE,
    oob_score=True,
    n_jobs=-1,
)
rf_s2.fit(X_train_s2, y_train_s2)
print(f"Stage 2 OOB score: {rf_s2.oob_score_:.4f}")
print(f"Training set: {len(train_s2_df):,} anomaly rows  "
      f"(attack={y_train_s2.sum():,}  fault={(~y_train_s2.astype(bool)).sum():,})")

p_s2 = rf_s2.predict_proba(X_test)[:, 1]  # P(attack | row)


Training Stage 2 RF (attack vs fault, anomaly rows only)...
Stage 2 OOB score: 1.0000
Training set: 159,132 anomaly rows  (attack=8,063  fault=151,069)


## 5. Stage 2 Standalone Evaluation (Upper Bound)

Evaluates Stage 2 on **anomaly-only test rows** — as if Stage 1 had perfect recall. This isolates Stage 2's classification ability and shows the ceiling for attack/fault discrimination. Any gap between standalone and cascade fault recall is attributable to Stage 1 misses.

In [6]:
anomaly_test_mask = y_test_orig > 0
y_test_anomaly    = y_test_orig[anomaly_test_mask]  # 1=attack, 2=fault
p_s2_anomaly      = p_s2[anomaly_test_mask]

# At threshold 0.5 for attack
s2_pred = np.where(p_s2_anomaly >= 0.5, 1, 2)
s2_attack_recall = (s2_pred[y_test_anomaly == 1] == 1).mean()
s2_fault_recall  = (s2_pred[y_test_anomaly == 2] == 2).mean()
s2_accuracy      = (s2_pred == y_test_anomaly).mean()

print("Stage 2 Standalone (on anomaly-only test rows):")
print(f"  Attack recall: {s2_attack_recall:.1%}")
print(f"  Fault recall:  {s2_fault_recall:.1%}")
print(f"  Accuracy:      {s2_accuracy:.1%}")
print(f"  N anomaly rows: {anomaly_test_mask.sum():,}  "
      f"(attack={( y_test_anomaly==1).sum():,}  fault={(y_test_anomaly==2).sum():,})")


Stage 2 Standalone (on anomaly-only test rows):
  Attack recall: 100.0%
  Fault recall:  98.8%
  Accuracy:      98.8%
  N anomaly rows: 37,467  (attack=1,914  fault=35,553)


## 6. Full Cascade Evaluation

Chains Stage 1 and Stage 2: rows flagged anomalous by Stage 1 are passed to Stage 2 for attack/fault classification. Evaluated at each FPR operating point from Section 3 — results show the realistic end-to-end system performance.

In [7]:
cascade_results = []

for budget in FPR_BUDGETS:
    if budget not in operating_points:
        continue
    row = operating_points[budget]
    t1  = row["threshold"]

    stage1_anomaly = p_s1 >= t1
    pred = np.zeros(len(y_test_orig), dtype=np.int8)
    if stage1_anomaly.sum() > 0:
        pred[stage1_anomaly] = np.where(p_s2[stage1_anomaly] >= 0.5, 1, 2)

    fp     = ((y_test_orig == 0) & (pred > 0)).sum()
    fpr    = fp / n_normal_test
    atk_r  = (pred[y_test_orig == 1] == 1).mean() if n_attack_test > 0 else 0
    flt_r  = (pred[y_test_orig == 2] == 2).mean() if n_fault_test  > 0 else 0
    cascade_results.append({
        "fpr_budget": f"{budget:.1%}", "threshold": round(t1, 3),
        "attack_recall": round(atk_r, 4), "fault_recall": round(flt_r, 4),
        "normal_fpr": round(fpr, 4),
    })

    # Cache the 2% FPR predictions for reuse in per-fault, confusion matrix, and figure cells
    if abs(budget - 0.020) < 1e-9:
        pred_2pct = pred.copy()
        t1_2pct   = t1

casc_df = pd.DataFrame(cascade_results)
print("Full Cascade Results:")
print(casc_df.to_string(index=False))
casc_df.to_csv(RESULTS_DIR / "cascade_results.csv", index=False)

Full Cascade Results:
fpr_budget  threshold  attack_recall  fault_recall  normal_fpr
      0.5%      0.631         0.7503        0.4764      0.0048
      1.0%      0.530         0.8845        0.6933      0.0099
      2.0%      0.418         0.9540        0.8087      0.0196


## 7. Per-Fault-Type Recall (at 2% FPR Budget)

Breaks down cascade fault recall by fault type. Reveals whether recall gaps are uniform or concentrated in specific fault modes — gradual faults (drift, bias) tend to score lower than abrupt ones (dropout, stuck-at) because their rolling feature signatures are harder to distinguish from normal variation.

In [8]:
per_fault_rows = []
if "pred_2pct" in dir() and fault_type_test is not None:
    fault_mask = y_test_orig == 2
    for ft in sorted(set(fault_type_test[fault_mask])):
        if not ft or (isinstance(ft, float) and np.isnan(ft)):
            continue
        ft_mask   = fault_mask & (fault_type_test == ft)
        n_ft      = ft_mask.sum()
        n_correct = (pred_2pct[ft_mask] == 2).sum()
        per_fault_rows.append({
            "fault_type": ft,
            "n_test_rows": int(n_ft),
            "recall": round(n_correct / n_ft, 4) if n_ft > 0 else 0,
        })

    pf_df = pd.DataFrame(per_fault_rows)
    print(f"Per-Fault-Type Recall (FPR budget=2.0%, t1={t1_2pct:.3f}):")
    print(pf_df.to_string(index=False))
    pf_df.to_csv(RESULTS_DIR / "per_fault_recall.csv", index=False)
else:
    print("pred_2pct not available — run cascade evaluation cell first.")

Per-Fault-Type Recall (FPR budget=2.0%, t1=0.418):
           fault_type  n_test_rows  recall
                 bias         5402  0.7873
                drift         9137  0.8130
 intermittent_dropout        13354  0.7301
precision_degradation         3895  0.9897
             stuck_at         3765  0.9201


### 7.1 Drift Recall Diagnostic

Investigates why drift recall is the lowest of all fault types. Compares Stage 1 `P(anomaly)` scores and scaled SP sensor values between drift test rows and normal rows — overlapping distributions indicate the rolling features are not separating slow drift from normal variation.

In [9]:
SP_SENSORS = ["2_FIC_101_SP", "2_FIC_201_SP", "2_FIC_401_SP"]
drift_mask  = (y_test_orig == 2) & (fault_type_test == "drift")
normal_mask = y_test_orig == 0

print(f"Test drift rows: {drift_mask.sum()}  |  Test normal rows: {normal_mask.sum():,}")
print(f"\nStage 1 P(anomaly) for drift rows:")
print(f"  min={p_s1[drift_mask].min():.4f}  max={p_s1[drift_mask].max():.4f}  "
      f"mean={p_s1[drift_mask].mean():.4f}  median={np.median(p_s1[drift_mask]):.4f}")
print(f"Stage 1 P(anomaly) for normal rows:")
print(f"  min={p_s1[normal_mask].min():.4f}  max={p_s1[normal_mask].max():.4f}  "
      f"mean={p_s1[normal_mask].mean():.4f}  median={np.median(p_s1[normal_mask]):.4f}")

print(f"\nSP sensor scaled values (drift vs normal):")
for sp in SP_SENSORS:
    if sp not in FEATURE_COLS:
        print(f"  {sp}: NOT in FEATURE_COLS (dropped)")
        continue
    idx = FEATURE_COLS.index(sp)
    drift_vals  = X_test[drift_mask, idx]
    normal_vals = X_test[normal_mask, idx]
    print(f"  {sp}:")
    print(f"    drift  — min={drift_vals.min():.3f}  max={drift_vals.max():.3f}  "
          f"mean={drift_vals.mean():.3f}  std={drift_vals.std():.3f}")
    print(f"    normal — min={normal_vals.min():.3f}  max={normal_vals.max():.3f}  "
          f"mean={normal_vals.mean():.3f}  std={normal_vals.std():.3f}")

print(f"\nfault_sensor breakdown for test drift rows:")
drift_test_df = test_df[drift_mask.astype(bool)]
if "fault_sensor" in drift_test_df.columns:
    print(drift_test_df["fault_sensor"].value_counts().to_string())
else:
    print("  fault_sensor column not in features parquet")

Test drift rows: 9137  |  Test normal rows: 144,118

Stage 1 P(anomaly) for drift rows:
  min=0.0000  max=0.9900  mean=0.6129  median=0.6500
Stage 1 P(anomaly) for normal rows:
  min=0.0000  max=0.9567  mean=0.0529  median=0.0167

SP sensor scaled values (drift vs normal):
  2_FIC_101_SP:
    drift  — min=-4.657  max=2.100  mean=-0.039  std=1.181
    normal — min=-1.022  max=2.100  mean=-0.034  std=0.968
  2_FIC_201_SP:
    drift  — min=-3.087  max=4.208  mean=0.173  std=1.243
    normal — min=-1.090  max=2.523  mean=-0.030  std=0.983
  2_FIC_401_SP:
    drift  — min=-4.759  max=4.260  mean=-0.038  std=1.390
    normal — min=-1.085  max=2.414  mean=-0.029  std=0.968

fault_sensor breakdown for test drift rows:
  fault_sensor column not in features parquet


## 8. FPR Budget Sensitivity Table

Tabulates cascade fault recall and attack recall across the three FPR operating points. Shows the recall–FPR tradeoff for different deployment sensitivity requirements.

In [10]:
print("FPR Budget Sensitivity:")
print(f"{'FPR Budget':>12}  {'Fault Recall':>13}  {'Attack Recall':>14}  {'Threshold':>10}")
for r in cascade_results:
    print(f"  {r['fpr_budget']:>8}  {r['fault_recall']:>12.1%}  {r['attack_recall']:>13.1%}  {r['threshold']:>10.3f}")


FPR Budget Sensitivity:
  FPR Budget   Fault Recall   Attack Recall   Threshold
      0.5%         47.6%          75.0%       0.631
      1.0%         69.3%          88.4%       0.530
      2.0%         80.9%          95.4%       0.418


## 9. Confusion Matrix (at 2% FPR Budget)

Row-normalized confusion matrix for the full cascade at the 2% FPR operating point. Misclassified fault rows appear as **Normal** (Stage 1 miss) or **Attack** (Stage 2 miss) — the relative counts identify which stage is the dominant source of error.

In [11]:
if "pred_2pct" not in dir():
    print("pred_2pct not available — run cascade evaluation cell first.")
else:
    cm = confusion_matrix(y_test_orig, pred_2pct, labels=[0, 1, 2])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    class_names = ["Normal", "Attack", "Fault"]

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Row-normalized fraction", fontsize=10)

    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(class_names, fontsize=11)
    ax.set_yticklabels(class_names, fontsize=11)
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_title(f"Cascade Confusion Matrix (FPR budget=2.0%, t={t1_2pct:.3f})", fontsize=12)

    for i in range(3):
        for j in range(3):
            pct = cm_norm[i, j]
            count = cm[i, j]
            color = "white" if pct > 0.5 else "black"
            ax.text(j, i, f"{pct:.1%}\n({count:,})", ha="center", va="center",
                    color=color, fontsize=10)

    fig.tight_layout()
    out = FIGURES_DIR / "confusion_matrix.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")
    print(f"\n{classification_report(y_test_orig, pred_2pct, target_names=class_names)}")

Saved: data/results/figures/confusion_matrix.png

              precision    recall  f1-score   support

      Normal       0.96      0.98      0.97    144118
      Attack       0.83      0.95      0.89      1914
       Fault       0.91      0.81      0.86     35553

    accuracy                           0.95    181585
   macro avg       0.90      0.91      0.90    181585
weighted avg       0.95      0.95      0.95    181585



## 10. Stage 1 ROC-Style Recall vs FPR Curve

Plots fault recall and attack recall as a function of normal FPR across all 500 threshold values. Vertical lines mark the three FPR budget operating points. The asymmetric recall behavior between fault and attack across the threshold range illustrates why Stage 1 is the cascade bottleneck for fault detection.

In [12]:
fig, ax = plt.subplots(figsize=(7, 5))
fpr_vals   = s1_df["fpr"].values
fault_vals = s1_df["fault_recall"].values
atk_vals   = s1_df["attack_recall"].values

ax.plot(fpr_vals, fault_vals, label="Fault Recall", color="firebrick", lw=1.5)
ax.plot(fpr_vals, atk_vals,   label="Attack Recall", color="steelblue", lw=1.5)

for budget in FPR_BUDGETS:
    ax.axvline(x=budget, color="gray", linestyle="--", lw=0.8, alpha=0.7)
    ax.text(budget + 0.001, 0.05, f"{budget:.1%}", fontsize=8, color="gray")

ax.set_xlabel("Normal FPR", fontsize=12)
ax.set_ylabel("Recall", fontsize=12)
ax.set_title("Stage 1 Recall vs FPR Tradeoff", fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 0.06)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

fig.tight_layout()
out = FIGURES_DIR / "stage1_recall_vs_fpr.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")


Saved: data/results/figures/stage1_recall_vs_fpr.png


## 11. Save Models & Results

Writes three artifacts to `data/`:

- **`models/stage1_rf.joblib`** — fitted Stage 1 RandomForest (normal vs anomaly)
- **`models/stage2_rf.joblib`** — fitted Stage 2 RandomForest (attack vs fault)
- **`results/final_results.json`** — OOB scores, cascade operating points, and standalone Stage 2 metrics

In [13]:
joblib.dump(rf_s1, MODELS_DIR / "stage1_rf.joblib")
joblib.dump(rf_s2, MODELS_DIR / "stage2_rf.joblib")
print(f"Models saved to {MODELS_DIR}/")

results_summary = {
    "stage1_oob_score":   round(rf_s1.oob_score_, 4),
    "stage2_oob_score":   round(rf_s2.oob_score_, 4),
    "stage2_standalone": {
        "attack_recall": round(s2_attack_recall, 4),
        "fault_recall":  round(s2_fault_recall, 4),
        "accuracy":      round(s2_accuracy, 4),
    },
    "cascade_operating_points": cascade_results,
}

(RESULTS_DIR / "final_results.json").write_text(
    json.dumps(results_summary, indent=2)
)
print(f"Saved: {RESULTS_DIR / 'final_results.json'}")
print("\nAll done.")

print(f"Completed: {datetime.now()}")


Models saved to data/models/
Saved: data/results/final_results.json

All done.
Completed: 2026-04-25 10:01:53.608678
